# Member 1: Data Preprocessing, KPI Engine & Exploratory Data Analysis (EDA)
**GenAI-Based Data Visualisation & Narrative Generator — Semester 7 Final Case Study**

### Module Scope & Objectives (Member 1 - Data & Analytics):
1. **Dataset Definition & Ingestion**: Load sales dataset, validate schema, inspect dimensions, data types, and nulls.
2. **Data Cleaning & Transformation**: Handle missing values, filter corrupt rows, normalise strings, and engineer derived fields (`profit_margin`, `period_label`, `year`, `quarter`, `month`).
3. **KPI Calculation Engine**: Compute headline metrics (`Total Revenue`, `Total Profit`, `Total Orders`, `AOV`, `Profit Margin %`, `PoP Growth`).
4. **Trend, Ranking & Anomaly Detection**: Statistical linear regression for time-series direction, category/region rankings, and IQR-based anomaly detection.
5. **Contract Export**: Build and serialize the structured analytics JSON consumed by Member 2.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set project root path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from backend.analytics.preprocessing import load_csv, validate_schema, inspect, clean, add_derived_fields, preprocess, get_column_catalogue
from backend.analytics.kpis import compute_all_kpis, total_revenue, total_profit, total_orders, average_order_value, profit_margin_pct, monthly_revenue
from backend.analytics.trends import detect_revenue_trend, category_comparison, region_comparison, product_ranking, detect_anomalies, describe_distribution, compute_all_trends_and_comparisons
from backend.analytics.analytics_builder import build_analytics_json

print("Member 1 Analytics Modules loaded successfully!")

## 1. Dataset Ingestion & Schema Inspection

In [ ]:
csv_path = project_root / "data" / "sales.csv"
raw_df = load_csv(csv_path)
validate_schema(raw_df)
raw_inspection = inspect(raw_df)

print(f"Raw Dataset Shape: {raw_inspection['shape']}")
print(f"Null Counts:\n{pd.Series(raw_inspection['null_counts'])}")
print(f"Duplicate Orders: {raw_inspection['duplicate_count']}")
raw_df.head()

## 2. Preprocessing & Feature Engineering
- Cleaning steps: Date parsing, numeric coercion, sanity bounds (>0 for sales/quantity), title-case normalization.
- Feature engineering: `year`, `quarter`, `month`, `month_name`, `week`, `profit_margin`, `revenue`, `period_label`.

In [ ]:
df_clean = preprocess(csv_path)
print(f"Cleaned & Preprocessed Shape: {df_clean.shape}")
print("Sample of derived fields:")
df_clean[["order_id", "date", "sales", "profit", "profit_margin", "quarter", "period_label"]].head()

## 3. Exploratory Data Analysis & Visualizations

In [ ]:
plt.figure(figsize=(14, 10))

# 1. Monthly Revenue Trend
plt.subplot(2, 2, 1)
monthly_df = monthly_revenue(df_clean)
plt.plot(monthly_df['month_name'], monthly_df['revenue'] / 1e6, marker='o', color='#2563eb', linewidth=2)
plt.title("Monthly Revenue (Millions INR)")
plt.xlabel("Month")
plt.ylabel("Revenue (M INR)")
plt.grid(True, linestyle='--', alpha=0.5)

# 2. Revenue by Category
plt.subplot(2, 2, 2)
cat_rev = df_clean.groupby('category')['sales'].sum().sort_values(ascending=False) / 1e6
cat_rev.plot(kind='bar', color='#10b981')
plt.title("Revenue by Category (Millions INR)")
plt.xlabel("Category")
plt.ylabel("Revenue (M INR)")
plt.xticks(rotation=30)

# 3. Revenue by Region
plt.subplot(2, 2, 3)
reg_rev = df_clean.groupby('region')['sales'].sum().sort_values(ascending=False) / 1e6
reg_rev.plot(kind='bar', color='#8b5cf6')
plt.title("Revenue by Region (Millions INR)")
plt.xlabel("Region")
plt.ylabel("Revenue (M INR)")
plt.xticks(rotation=30)

# 4. Profit Margin Distribution
plt.subplot(2, 2, 4)
plt.hist(df_clean['profit_margin'].dropna(), bins=30, color='#f59e0b', edgecolor='black', alpha=0.7)
plt.title("Profit Margin Distribution (%)")
plt.xlabel("Profit Margin (%)")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

## 4. Headline KPI Calculations & Validation
All formulas match specifications defined in the case study document:
- Total Revenue: $\sum \text{sales}$
- Total Profit: $\sum \text{profit}$
- Average Order Value: $\frac{\text{Total Revenue}}{\text{Total Orders}}$
- Profit Margin: $\frac{\text{Total Profit}}{\text{Total Revenue}} \times 100$

In [ ]:
kpi_results = compute_all_kpis(df_clean)
kpi_df = pd.DataFrame(kpi_results["metrics"])
kpi_df[["name", "value", "unit", "description"]]

## 5. Statistical Trend, Comparison & Anomaly Detection
- **Trend Detection**: Linear regression slope on monthly aggregates.
- **Anomaly Detection**: Interquartile Range (IQR) on daily sales ($[Q_1 - 1.5\cdot IQR, Q_3 + 1.5\cdot IQR]$).

In [ ]:
trends_and_comparisons = compute_all_trends_and_comparisons(df_clean)

print("=== REVENUE TREND ===")
print(trends_and_comparisons["trends"])

print("\n=== TOP CATEGORY & REGION COMPARISONS ===")
for comp in trends_and_comparisons["comparisons"][:2]:
    print(f"{comp['dimension'].capitalize()}: Highest -> {comp['highest']['name']} ({comp['highest']['value']:,}), Lowest -> {comp['lowest']['name']} ({comp['lowest']['value']:,})")

print(f"\n=== DETECTED ANOMALIES (Total: {len(trends_and_comparisons['anomalies'])}) ===")
if trends_and_comparisons["anomalies"]:
    display(pd.DataFrame(trends_and_comparisons["anomalies"]).head())
else:
    print("No extreme daily anomalies detected beyond IQR fences.")

## 6. End-to-End JSON Contract Generation (Member 1 Output)

In [ ]:
final_output = build_analytics_json(csv_path)
print(f"Analytics JSON top-level keys: {list(final_output.keys())}")
print(f"Dataset: {final_output['dataset']['name']} ({final_output['dataset']['record_count']} rows)")
print(f"Metrics Count: {len(final_output['metrics'])}")
print(f"Visualizations Included: {len(final_output['visualizations'])}")